# 00_Data overview and sanity checks
Converts the three KuaiRand-Pure logs to Parquet, profiles their size and schema, checks that treatment (random exposure) and control (recommended) are comparable, and compares raw engagement metrics before the formal analysis.

## 1. Load data

Converts the three raw CSV logs to Parquet in `data/processed/`. The conversion is skipped automatically if a Parquet file already exists.

| Name | Raw file | Role |
|---|---|---|
| `pre` | `log_standard_4_08_to_4_21_pure` | Pre-period, used later as the CUPED covariate |
| `control` | `log_standard_4_22_to_5_08_pure` | Recommended videos (`is_rand = 0`) |
| `treatment` | `log_random_4_22_to_5_08_pure` | Randomly exposed videos (`is_rand = 1`) |

In [1]:
import os
import duckdb

duckdb.sql("SET enable_progress_bar = false")
raw_dir = '../data/raw/KuaiRand-Pure/data'
processed_dir = '../data/processed'
os.makedirs(processed_dir, exist_ok=True)

logs = {
    'pre':       'log_standard_4_08_to_4_21_pure',
    'control':   'log_standard_4_22_to_5_08_pure',
    'treatment': 'log_random_4_22_to_5_08_pure',
}

# Skip if the Parquet file already exists; convert only on the first run
for name, file in logs.items():
    parquet_path = f'{processed_dir}/{name}.parquet'
    if os.path.exists(parquet_path):
        print(f'{name}: Parquet already exists, skipping conversion.')
    else:
        duckdb.sql(f"""
        COPY (SELECT * FROM read_csv_auto('{raw_dir}/{file}.csv'))
        TO '{parquet_path}' (FORMAT PARQUET)
        """)
        print(f'{name}: conversion done.')

pre: Parquet already exists, skipping conversion.
control: Parquet already exists, skipping conversion.
treatment: Parquet already exists, skipping conversion.


### Register the `pre`, `control` and `treatment` views
Creates one view per Parquet file so every query can use `FROM pre`, `FROM control` or `FROM treatment`. Re-run this cell after a kernel restart.

In [2]:
duckdb.sql("""
CREATE OR REPLACE VIEW pre       AS SELECT * FROM '../data/processed/pre.parquet';
CREATE OR REPLACE VIEW control   AS SELECT * FROM '../data/processed/control.parquet';
CREATE OR REPLACE VIEW treatment AS SELECT * FROM '../data/processed/treatment.parquet';
""")

## 2. Data profiling

### Row counts
Number of impressions in each log.

In [3]:
duckdb.sql("""
SELECT 'pre' AS log, COUNT(*) AS n FROM pre
UNION ALL
SELECT 'control', COUNT(*) FROM control
UNION ALL
SELECT 'treatment', COUNT(*) FROM treatment
""").show()

┌───────────┬─────────┐
│    log    │    n    │
│  varchar  │  int64  │
├───────────┼─────────┤
│ pre       │ 1141112 │
│ control   │  295497 │
│ treatment │ 1186059 │
└───────────┴─────────┘



### Schema check
All three logs share the same 19 columns; `treatment` is shown.

In [4]:
duckdb.sql("DESCRIBE SELECT * FROM treatment").show()

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ user_id           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ video_id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ date              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ hourmin           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ time_ms           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_click          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_like           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_follow         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_comment        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_forward

**Key findings: data profiling**
- Treatment has 1,186,059 impressions, matching the count documented in the KuaiRand README.
- Control has only 295,497 impressions in the same period. The Pure version keeps standard-log rows only for the 7,583 candidate-pool videos, so control is about a quarter the size of treatment.
- The pre-period has 1,141,112 impressions, enough to build a pre-period covariate for most users.
- All 19 columns are `BIGINT`, including the 0/1 engagement flags and `play_time_ms`.

## 3. Group comparability

### Combine treatment and control into `exp`
Stacks both logs into one view with a `grp` label, so each comparison below is a single `GROUP BY grp`.

In [5]:
duckdb.sql("""
CREATE OR REPLACE VIEW exp AS
SELECT *, 'treatment' AS grp FROM treatment
UNION ALL
SELECT *, 'control' AS grp FROM control
""")

### Group assignment check
Expect `is_rand = 1` for every treatment row, `is_rand = 0` for every control row, and identical date ranges.

In [ ]:
duckdb.sql("""
SELECT grp, is_rand,
       COUNT(*) AS n,
       COUNT(DISTINCT user_id) AS users,
       MIN(date) AS start_date,
       MAX(date) AS end_date
FROM exp
GROUP BY grp, is_rand
ORDER BY grp
""").show()

┌───────────┬─────────┬─────────┬───────┬────────────┬──────────┐
│    grp    │ is_rand │    n    │ users │ start_date │ end_date │
│  varchar  │  int64  │  int64  │ int64 │   int64    │  int64   │
├───────────┼─────────┼─────────┼───────┼────────────┼──────────┤
│ control   │       0 │  295497 │ 25877 │   20220422 │ 20220508 │
│ treatment │       1 │ 1186059 │ 27285 │   20220422 │ 20220508 │
└───────────┴─────────┴─────────┴───────┴────────────┴──────────┘



### Tab distribution
`is_click` means *click* in the two-column UI but *valid play* in the single-column UI, so the same field differs by `tab`. `FILTER (WHERE ...)` counts each group separately in one pass; `SUM(...) OVER ()` gives each group's total as the denominator.

In [ ]:
duckdb.sql("""
SELECT tab,
       COUNT(*) FILTER (WHERE grp = 'treatment') AS treat_n,
       COUNT(*) FILTER (WHERE grp = 'control')   AS ctrl_n,
       ROUND(100.0 * COUNT(*) FILTER (WHERE grp = 'treatment')
             / SUM(COUNT(*) FILTER (WHERE grp = 'treatment')) OVER (), 2) AS treat_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE grp = 'control')
             / SUM(COUNT(*) FILTER (WHERE grp = 'control')) OVER (), 2)   AS ctrl_pct
FROM exp
GROUP BY tab
ORDER BY tab
""").show()

┌───────┬─────────┬────────┬───────────┬──────────┐
│  tab  │ treat_n │ ctrl_n │ treat_pct │ ctrl_pct │
│ int64 │  int64  │ int64  │  double   │  double  │
├───────┼─────────┼────────┼───────────┼──────────┤
│     0 │       0 │  28074 │       0.0 │      9.5 │
│     1 │ 1178025 │ 223513 │     99.32 │    75.64 │
│     2 │     764 │   9073 │      0.06 │     3.07 │
│     3 │       0 │    440 │       0.0 │     0.15 │
│     4 │       0 │  17641 │       0.0 │     5.97 │
│     5 │       0 │    767 │       0.0 │     0.26 │
│     6 │       0 │  13114 │       0.0 │     4.44 │
│     7 │       0 │    171 │       0.0 │     0.06 │
│     8 │       0 │   1358 │       0.0 │     0.46 │
│     9 │       0 │    192 │       0.0 │     0.06 │
│    10 │       0 │     72 │       0.0 │     0.02 │
│    11 │    6755 │    308 │      0.57 │      0.1 │
│    12 │       0 │    638 │       0.0 │     0.22 │
│    13 │       0 │    116 │       0.0 │     0.04 │
│    14 │     515 │     20 │      0.04 │     0.01 │
└───────┴───

**Key findings: group comparability**
- Groups are cleanly separated by `is_rand` and cover the same 17-day window (2022-04-22 to 2022-05-08).
- Treatment reaches all 27,285 users; control reaches 25,877.
- Random exposures appear only in tabs 1, 2, 11 and 14, and 99.3% are in tab 1. Control spans all 15 tabs, with 75.6% in tab 1.
- **Scope:** because the tab mix differs and `is_click` changes meaning across UIs, the formal analysis is restricted to **tab 1**, where both groups have ample volume (1.18M vs. 224K impressions).

## 4. Raw metric comparison

### All tabs (preliminary)
Unadjusted impression-level means. Not a valid comparison on its own because the tab mix differs; shown for reference.

In [ ]:
duckdb.sql("""
SELECT grp,
       ROUND(AVG(is_click), 4)            AS click_rate,
       ROUND(AVG(long_view), 4)           AS long_view_rate,
       ROUND(AVG(play_time_ms) / 1000, 2) AS avg_play_sec,
       ROUND(AVG(is_like), 5)             AS like_rate,
       ROUND(AVG(is_hate), 5)             AS hate_rate
FROM exp
GROUP BY grp
""").show()

┌───────────┬────────────┬────────────────┬──────────────┬───────────┬───────────┐
│    grp    │ click_rate │ long_view_rate │ avg_play_sec │ like_rate │ hate_rate │
│  varchar  │   double   │     double     │    double    │  double   │  double   │
├───────────┼────────────┼────────────────┼──────────────┼───────────┼───────────┤
│ control   │      0.445 │         0.3134 │        21.47 │   0.01771 │   0.00078 │
│ treatment │     0.1762 │          0.085 │         6.94 │    0.0048 │   0.00114 │
└───────────┴────────────┴────────────────┴──────────────┴───────────┴───────────┘



### Tab 1 only, with relative lift
Same metrics restricted to tab 1. `lift_pct` is the treatment mean relative to the control mean; negative values show how much worse a random impression is than a recommended one.

In [ ]:
duckdb.sql("""
WITH m AS (
    -- one row per group with the tab-1 mean of each metric
    SELECT grp,
           COUNT(*) AS n,
           AVG(is_click)              AS click_rate,
           AVG(long_view)             AS long_view_rate,
           AVG(play_time_ms) / 1000.0 AS avg_play_sec,
           AVG(is_like)               AS like_rate,
           AVG(is_hate)               AS hate_rate
    FROM exp
    WHERE tab = 1
    GROUP BY grp
)
-- put treatment and control side by side and compute the relative difference
SELECT t.n AS treat_n,
       c.n AS ctrl_n,
       ROUND(100 * (t.click_rate     / c.click_rate     - 1), 1) AS click_lift_pct,
       ROUND(100 * (t.long_view_rate / c.long_view_rate - 1), 1) AS long_view_lift_pct,
       ROUND(100 * (t.avg_play_sec   / c.avg_play_sec   - 1), 1) AS play_sec_lift_pct,
       ROUND(100 * (t.like_rate      / c.like_rate      - 1), 1) AS like_lift_pct,
       ROUND(100 * (t.hate_rate      / c.hate_rate      - 1), 1) AS hate_lift_pct
FROM m t, m c
WHERE t.grp = 'treatment'
  AND c.grp = 'control'
""").show()

┌─────────┬────────┬────────────────┬────────────────────┬───────────────────┬───────────────┬───────────────┐
│ treat_n │ ctrl_n │ click_lift_pct │ long_view_lift_pct │ play_sec_lift_pct │ like_lift_pct │ hate_lift_pct │
│  int64  │ int64  │     double     │       double       │      double       │    double     │    double     │
├─────────┼────────┼────────────────┼────────────────────┼───────────────────┼───────────────┼───────────────┤
│ 1178025 │ 223513 │          -64.4 │              -75.7 │             -71.6 │         -76.0 │          37.6 │
└─────────┴────────┴────────────────┴────────────────────┴───────────────────┴───────────────┴───────────────┘



**Key findings:**
- Tab 1 lift (random vs. recommended): click −64.4%, long view −75.7%, play time −71.6%, like −76.0%, hate +37.6%.
- The tab-1 costs are 3–4 points larger than the all-tab figures (e.g. long view −75.7% vs. −72.9%): recommendations perform better in tab 1, so mixing in other tabs dilutes the control mean and understates the cost of random exposure.
- Losses are largest on deep-engagement metrics (long view, like) and smallest on click: random videos still get opened, but rarely watched through or liked.